# Domain Adaptation through QLoRA

LoRA:
Freeze the base model and train low-rank weight updates.

QLoRA:
Store supported base-model weights in a quantized representation,
freeze them, and train floating-point LoRA updates.

Conceptually:

Output = Quantized-base transformation + LoRA correction

Important:
- 4-bit storage does not mean all computations use 4-bit arithmetic.
- LoRA parameters remain floating point.
- Embeddings and some other components may remain unquantized.
- Total training memory includes activations, gradients, and optimizer state.

This notebook provides:
1. A conceptual CPU quantization exercise.
2. Actual 4-bit LoRA training on a supported NVIDIA GPU.

In [1]:
%pip install -q torch "transformers>=4.45,<5" "peft>=0.13,<0.19" accelerate pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
# If you have GPU uncomment this and run

%pip install -q "bitsandbytes>=0.43.3"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import math
import random

import pandas as pd
import torch

from IPython.display import display

# Choose "cpu", "cuda", or "auto".
DEVICE_CHOICE = "auto"

if DEVICE_CHOICE == "auto":
    RUN_GPU = torch.cuda.is_available()

elif DEVICE_CHOICE == "cuda":
    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is unavailable. Choose CPU for the conceptual "
            "exercise or use a supported GPU environment."
        )
    RUN_GPU = True

elif DEVICE_CHOICE == "cpu":
    RUN_GPU = False

else:
    raise ValueError('Choose "cpu", "cuda", or "auto".')

random.seed(42)
torch.manual_seed(42)

if RUN_GPU:
    torch.cuda.manual_seed_all(42)
    device = torch.device("cuda:0")
    print("GPU training path:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("CPU conceptual path — no QLoRA model training.")

GPU training path: NVIDIA GeForce RTX 5090 Laptop GPU


In [3]:
# CPU-friendly quantization 

# It maps weights onto 16 uniformly spaced levels. This illustrates quantization, 
# but it is not NF4, double quantization, or a packed 4-bit implementation.

The CPU path illustrates quantization; the GPU path performs QLoRA training.

In [2]:
weights = torch.tensor([
    -0.95, -0.72, -0.43, -0.18,
     0.06,  0.29,  0.57,  0.91,
], dtype=torch.float32)

BITS = 4 # Using 4 bits, which can represent 16 different codes.
LEVELS = 2 ** BITS

# Find the range of the example weights.
minimum = weights.min()
maximum = weights.max()
# Calculate the distance between uniformly spaced quantization levels.
scale = (maximum - minimum) / (LEVELS - 1)


# Map each weight to its nearest integer code between 0 and 15.
# We store the codes in uint8 here for demonstration.
codes = torch.round(
    (weights - minimum) / scale
).clamp(0, LEVELS - 1).to(torch.uint8)

reconstructed = codes.float() * scale + minimum # Reconstruct approximate floating-point weights from the codes.
# This is a uniform quantization demonstration.
# It is not the NF4 quantization used in the GPU training section.
# uint8 uses one byte per code here; packed 4-bit storage uses half a byte.

display(pd.DataFrame({
    "original_weight": weights.numpy(),
    "quantization_code": codes.numpy(),
    "reconstructed_weight": reconstructed.numpy(),
    "absolute_error": (weights - reconstructed).abs().numpy(),
}))

print("Number of available levels:", LEVELS)
print(
    "Mean absolute reconstruction error:",
    (weights - reconstructed).abs().mean().item(),
)

print("\nThese codes occupy uint8 storage here.")
print("Actual packed 4-bit storage can place two codes in one byte.")

,original_weight,quantization_code,reconstructed_weight,absolute_error
0,-0.95,0,-0.950,0.000000e+00
1,-0.72,2,-0.702,1.800001e-02
2,-0.43,4,-0.454,2.399999e-02
3,-0.18,6,-0.206,2.600002e-02
4,0.06,8,0.042,1.800000e-02
5,0.29,10,0.290,2.980232e-08
6,0.57,12,0.538,3.200006e-02
7,0.91,15,0.910,0.000000e+00


Number of available levels: 16
Mean absolute reconstruction error: 0.014750014990568161

These codes occupy uint8 storage here.
Actual packed 4-bit storage can place two codes in one byte.


## quantization reduces storage precision and introduces reconstruction error. Training a LoRA correction can adapt the quantized model, but does not guarantee recovery of every quantization error.

In [5]:
# ompare theoretical weight storage
# These are weight-only estimates, excluding quantization metadata and 
# all other training memory.

In [3]:
PARAMETER_COUNT = 1_000_000

storage_table = pd.DataFrame([
    {
        "representation": "FP32",
        "bits_per_weight": 32,
        "ideal_weight_storage_MiB": PARAMETER_COUNT * 32 / 8 / 1024**2,
    },
    {
        "representation": "FP16",
        "bits_per_weight": 16,
        "ideal_weight_storage_MiB": PARAMETER_COUNT * 16 / 8 / 1024**2,
    },
    {
        "representation": "Packed 4-bit",
        "bits_per_weight": 4,
        "ideal_weight_storage_MiB": PARAMETER_COUNT * 4 / 8 / 1024**2,
    },
])

display(storage_table)

,representation,bits_per_weight,ideal_weight_storage_MiB
0,FP32,32,3.814697
1,FP16,16,1.907349
2,Packed 4-bit,4,0.476837


In [7]:
# CPU participants can finish here and use the preceding 05_ LoRA notebook for actual training. 
# The following cells run training only when RUN_GPU=True.

In [8]:
# GPU settings and domain data

In [4]:
MODEL_NAME = "/home/ravin/aidev/models/distilgpt2"

LORA_RANK = 4
LORA_ALPHA = 8

MAX_LENGTH = 96
BATCH_SIZE = 1
TRAINING_STEPS = 30
LEARNING_RATE = 5e-4

train_texts = [
    "Predictive maintenance uses equipment condition data to plan "
    "maintenance before a failure occurs.",

    "Preventive maintenance follows a planned schedule based on "
    "time or equipment usage.",

    "Vibration monitoring helps engineers identify changes in "
    "the condition of rotating machinery.",

    "Bearing faults can produce characteristic vibration patterns. "
    "Engineers examine frequency information and operating conditions.",

    "An increase in bearing temperature may indicate lubrication "
    "problems, increased load, or other developing issues.",

    "Root mean square vibration summarizes the magnitude of a "
    "vibration signal over a measurement interval.",

    "A condition monitoring baseline describes normal equipment "
    "behavior under known operating conditions.",

    "Sensor trends should be compared with operating speed and load "
    "because these factors can influence measurements.",

    "A single abnormal reading does not confirm a machine fault. "
    "Additional measurements help support a diagnosis.",

    "Lubrication reduces friction and wear between moving surfaces. "
    "Incorrect lubrication can contribute to bearing damage.",

    "Shaft misalignment can increase vibration and mechanical stress "
    "in rotating equipment.",

    "Maintenance decisions should combine sensor measurements, "
    "inspection findings, and equipment history.",
]

eval_texts = [
    "Condition-based maintenance uses observations of machine health "
    "to decide when maintenance is needed.",

    "A technician investigating elevated vibration should consider "
    "changes in load, speed, alignment, and bearing condition.",

    "Rising bearing temperature is a symptom that requires further "
    "investigation rather than an immediate fault diagnosis.",

    "Comparing measurements with a normal operating baseline helps "
    "identify changes in equipment condition.",
]

assert set(train_texts).isdisjoint(eval_texts)

In [10]:
# Load the local model in 4-bit
# The Transformers integration supports converting GPT-2’s Conv1D projections 
# into bitsandbytes quantized linear layers.

In [5]:
if RUN_GPU:
    import bitsandbytes as bnb

    from transformers import (
        AutoTokenizer,
        AutoModelForCausalLM,
        BitsAndBytesConfig,
    )

    from peft import (
        LoraConfig,
        TaskType,
        get_peft_model,
        prepare_model_for_kbit_training,
    )
    # Choose the floating-point computation type supported by yout GPU.
    compute_dtype = (
        torch.bfloat16
        if torch.cuda.is_bf16_supported()
        else torch.float16
    )
    # Describe how eligible base-model layers should be quantized.
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,# Store eligible linear-layer weights in packed 4-bit form.
        bnb_4bit_quant_type="nf4", # Use NormalFloat4: a nonuniform 4-bit quantization format.
        bnb_4bit_use_double_quant=True,# Quantize the quantization constants too, reducing metadata storage.
        # Perform the quantized layers' computation in this floating-point type.
        # Four-bit weight storage does not mean four-bit arithmetic.
        bnb_4bit_compute_dtype=compute_dtype,
    )

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        local_files_only=True,
    )
    tokenizer.pad_token = tokenizer.eos_token

    # Load the local base model using the quantization configuration.
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        local_files_only=True,# Read the model from local files.
        quantization_config=quantization_config,# Apply 4-bit quantization to eligible layers during loading.
        device_map={"": 0},# Place the model on GPU 0.
        torch_dtype=compute_dtype, # Set the loading dtype for floating-point components.
        attn_implementation="eager",# Use the eager attention implementation.
    )

    base_model.config.pad_token_id = tokenizer.pad_token_id# Specify the padding token used by our batches.
    base_model.config.use_cache = False # Disable the generation cache during training.
    # Count layers that were actually converted into 4-bit linear layers.
    quantized_layer_count = sum(
        isinstance(module, bnb.nn.Linear4bit)
        for module in base_model.modules()
    )

    assert quantized_layer_count > 0

    print("4-bit linear layers:", quantized_layer_count)
    print("Compute dtype:", compute_dtype)
    print(
        "Loaded model footprint:",
        f"{base_model.get_memory_footprint() / 1024**2:.2f} MiB",
    )

else:
    print("Skipped: GPU-only model loading.")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

4-bit linear layers: 24
Compute dtype: torch.bfloat16
Loaded model footprint: 95.49 MiB


| Setting | Purpose |
 
| `load_in_4bit=True` | Quantizes supported layers |
| `quant_type="nf4"` | Uses the NF4 quantization format |
| `use_double_quant=True` | Also quantizes quantization constants |
| `compute_dtype` | Selects floating-point computation precision |

In [12]:
# Prepare the model and attach LoRA

# After quantization, our target projections are Linear4bit layers,
# so use fan_in_fan_out=False here.

In [6]:
# Freeze the base-model parameters and prepare floating-point components
# for training alongside quantized layers.
# Disable gradient checkpointing for this small exercise.

if RUN_GPU:
    base_model = prepare_model_for_kbit_training(
        base_model,
        use_gradient_checkpointing=False,
    )
    # Configure the trainable LoRA branches.
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,# Train using next-token prediction.
        r=LORA_RANK, # Use an intermediate rank of 4 for matrices A and B.
        lora_alpha=LORA_ALPHA, # Scale the LoRA correction by alpha / rank.
        lora_dropout=0.0,# Apply no dropout in the LoRA branch.
        target_modules=["c_attn"],# Target DistilGPT2's combined query/key/value projection. for QWEN - ["q_proj", "v_proj"].
        fan_in_fan_out=False,# The quantized projections now use Linear4bit weight conventions.
        bias="none", # Keep original biases frozen.
    )
    # Add floating-point LoRA matrices to the selected quantized projections.
    # The quantized base weights remain frozen.
    model = get_peft_model(base_model, lora_config)

    model.print_trainable_parameters()# Display the trainable parameter count.
    # Verify that only LoRA parameters are trainable.
    assert all(
        "lora_" in name
        for name, parameter in model.named_parameters()
        if parameter.requires_grad
    )

    display(pd.DataFrame([
        {
            "parameter": name,
            "shape": str(tuple(parameter.shape)),
            "dtype": str(parameter.dtype),
        }
        for name, parameter in model.named_parameters()
        if parameter.requires_grad
    ]))

else:
    print("Skipped: GPU-only LoRA setup.")

trainable params: 73,728 || all params: 81,986,304 || trainable%: 0.0899


,parameter,shape,dtype
0,base_model.model.transformer.h.0.attn.c_attn.l...,"(4, 768)",torch.float32
1,base_model.model.transformer.h.0.attn.c_attn.l...,"(2304, 4)",torch.float32
2,base_model.model.transformer.h.1.attn.c_attn.l...,"(4, 768)",torch.float32
3,base_model.model.transformer.h.1.attn.c_attn.l...,"(2304, 4)",torch.float32
4,base_model.model.transformer.h.2.attn.c_attn.l...,"(4, 768)",torch.float32
5,base_model.model.transformer.h.2.attn.c_attn.l...,"(2304, 4)",torch.float32
6,base_model.model.transformer.h.3.attn.c_attn.l...,"(4, 768)",torch.float32
7,base_model.model.transformer.h.3.attn.c_attn.l...,"(2304, 4)",torch.float32
8,base_model.model.transformer.h.4.attn.c_attn.l...,"(4, 768)",torch.float32
9,base_model.model.transformer.h.4.attn.c_attn.l...,"(2304, 4)",torch.float32


In [14]:
## The preparation function freezes base parameters and performs numerical preparation for low-bit training.

In [15]:
## Prepare data and evaluation functions

In [7]:
if RUN_GPU:
    from torch.utils.data import DataLoader

    def collate_texts(texts):
        batch = tokenizer(
            [text + tokenizer.eos_token for text in texts],
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )

        labels = batch["input_ids"].clone()
        labels[batch["attention_mask"] == 0] = -100
        batch["labels"] = labels

        return batch

    train_loader = DataLoader(
        train_texts,
        batch_size=BATCH_SIZE,
        shuffle=True,
        collate_fn=collate_texts,
    )

    eval_loader = DataLoader(
        eval_texts,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_texts,
    )

    @torch.no_grad()
    def evaluate_loss():
        model.eval()

        loss_sum = 0.0
        token_count = 0

        for batch in eval_loader:
            batch = {
                key: value.to(device)
                for key, value in batch.items()
            }

            output = model(**batch)

            valid_tokens = (
                batch["labels"][:, 1:] != -100
            ).sum().item()

            loss_sum += output.loss.item() * valid_tokens
            token_count += valid_tokens

        return loss_sum / token_count

    @torch.no_grad()
    def complete_text(prompt):
        model.eval()

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
        ).to(device)

        output_ids = model.generate(
            **inputs,
            max_new_tokens=45,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

        prompt_length = inputs["input_ids"].shape[1]

        return tokenizer.decode(
            output_ids[0, prompt_length:],
            skip_special_tokens=True,
        ).strip()

    print("Data and evaluation functions ready.")

else:
    print("Skipped: GPU-only data setup.")

Data and evaluation functions ready.


In [17]:
# Record the quantized baseline

# This baseline is the quantized model before LoRA training, 
# rather than the full-precision baseline from Notebook 06.

In [8]:
if RUN_GPU:
    import hashlib

    def frozen_fingerprints():
        return {
            name: hashlib.sha256(
                parameter.detach().cpu().contiguous().numpy().tobytes()
            ).hexdigest()
            for name, parameter in model.named_parameters()
            if not parameter.requires_grad
        }
    # Record fingerprints of frozen parameter tensors before training.
    frozen_before = frozen_fingerprints()
    # Save independent copies of the initial LoRA matrices.
    lora_before = {
        name: parameter.detach().cpu().clone()
        for name, parameter in model.named_parameters()
        if parameter.requires_grad
    }
    # Evaluate the quantized model before updating LoRA.
    # This is a quantized baseline, not a full-precision baseline.
    before_loss = evaluate_loss()

    test_prompts = [
        "Predictive maintenance is",
        "When bearing temperature increases,",
        "Vibration monitoring helps engineers",
    ]

    before_answers = {
        prompt: complete_text(prompt)
        for prompt in test_prompts
    }

    print(f"Quantized baseline loss: {before_loss:.4f}")

else:
    print("Skipped: GPU-only baseline.")

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Quantized baseline loss: 5.5552


In [19]:
# Train LoRA on the quantized model
## We use ordinary AdamW for this small exercise

In [9]:
if RUN_GPU:
    # Collect only trainable LoRA parameters.
    trainable_parameters = [
        parameter
        for parameter in model.parameters()
        if parameter.requires_grad
    ]
    # Optimize these parameters using ordinary AdamW.
    # This notebook does not use a paged optimizer.
    optimizer = torch.optim.AdamW(
        trainable_parameters,
        lr=LEARNING_RATE,
        weight_decay=0.0,
    )

    iterator = iter(train_loader)
    training_history = []

    torch.cuda.reset_peak_memory_stats(device)

    model.train()

    for step in range(1, TRAINING_STEPS + 1):
        try:
            batch = next(iterator)
        except StopIteration:
            iterator = iter(train_loader)
            batch = next(iterator)

        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }
        # Inside the training loop:
        
        # Clear gradients from the previous step.
        optimizer.zero_grad(set_to_none=True)
        # Run the quantized model with its floating-point LoRA branches.
        # Quantized weights are used through floating-point computation.
        # Each targeted projection adds the correction scaling * B(A(input)).
        loss = model(**batch).loss
        # Backpropagate to calculate gradients for the LoRA matrices.
        # The stored quantized base weights remain frozen.
        loss.backward()
        # Limit gradient magnitude to help stabilize training.
        torch.nn.utils.clip_grad_norm_(
            trainable_parameters,
            max_norm=1.0,
        )

        optimizer.step()# Update only the trainable LoRA matrices.

        training_history.append({
            "step": step,
            "training_loss": loss.item(),
        })

        if step == 1 or step % 5 == 0 or step == TRAINING_STEPS:
            print(
                f"Step {step}/{TRAINING_STEPS} "
                f"| Loss: {loss.item():.4f}"
            )

    print(
        "\nPeak allocated GPU memory during training:",
        f"{torch.cuda.max_memory_allocated(device) / 1024**2:.2f} MiB",
    )

else:
    print("Skipped: CPU path does not perform QLoRA training.")

Step 1/30 | Loss: 5.4570
Step 5/30 | Loss: 6.2534
Step 10/30 | Loss: 6.1498
Step 15/30 | Loss: 4.9490
Step 20/30 | Loss: 6.4895
Step 25/30 | Loss: 5.8042
Step 30/30 | Loss: 6.4758

Peak allocated GPU memory during training: 258.91 MiB


In [21]:
# Compare results and verify updates

In [10]:
if RUN_GPU:
    after_loss = evaluate_loss()

    display(pd.DataFrame([
        {
            "stage": "Quantized model before training",
            "evaluation_loss": before_loss,
            "perplexity": math.exp(before_loss),
        },
        {
            "stage": "Quantized model with trained LoRA",
            "evaluation_loss": after_loss,
            "perplexity": math.exp(after_loss),
        },
    ]))

    for prompt in test_prompts:
        print("\nPROMPT:", prompt)
        print("\nBEFORE:", before_answers[prompt])
        print("\nAFTER:", complete_text(prompt))
        print("=" * 70)

    frozen_after = frozen_fingerprints()# Recalculate fingerprints of the frozen parameter tensors.
    # Confirm that the frozen parameter values stayed identical.
    assert frozen_before == frozen_after, (
        "A frozen parameter unexpectedly changed."
    )
    # Find LoRA tensors whose values changed during training.
    changed_lora = [
        name
        for name, parameter in model.named_parameters()
        if parameter.requires_grad
        and not torch.equal(
            parameter.detach().cpu(),
            lora_before[name],
        )
    ]

    assert len(changed_lora) > 0# Confirm that training changed at least one LoRA tensor.

    print("\nChanged LoRA tensors:", len(changed_lora))
    print("Verified: frozen base parameters stayed identical.")

else:
    print("CPU conceptual exercise completed.")

,stage,evaluation_loss,perplexity
0,Quantized model before training,5.555188,258.575662
1,Quantized model with trained LoRA,5.085584,161.674307



PROMPT: Predictive maintenance is

BEFORE: a key component of the human brain. It is important to understand the role of the brain in the development of cognitive and emotional development.

AFTER: a critical component of the maintenance of the system.

PROMPT: When bearing temperature increases,

BEFORE: the body is more sensitive to temperature than the body.

AFTER: the temperature of the body increases.

PROMPT: Vibration monitoring helps engineers

BEFORE: to better understand the impact of the stress on the brain.

AFTER: to detect and diagnose a problem.


TypeError: Got unsupported ScalarType BFloat16

In [23]:
# QLoRA saves memory by storing eligible base-model weights in 4-bit form. 
# During training, we keep those weights frozen and update only the floating-point LoRA matrices

## Exercises

1. Why does 4-bit quantization provide 16 possible codes?

2. Why do reconstructed weights differ from the original weights?

3. What is the difference between storage dtype and compute dtype?

4. Which parameters are trained in QLoRA?

5. Why does total training memory exceed the size of the stored weights?

6. GPU participants:
   Compare ranks 4 and 8 using a fresh model each time.

7. Compare LoRA and QLoRA.
   Is lower memory guaranteed to produce better accuracy?

## Main takeaway

LoRA reduces the number of trainable parameters.

QLoRA additionally reduces storage precision for supported
frozen base-model weights.

Our CPU exercise illustrated quantization.
Our GPU exercise trained LoRA updates on a 4-bit NF4 base model.